In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/digit-recognizer/sample_submission.csv
/kaggle/input/competitions/digit-recognizer/train.csv
/kaggle/input/competitions/digit-recognizer/test.csv


In [5]:

# --- Load ---
train = pd.read_csv("/kaggle/input/competitions/digit-recognizer/train.csv")
y = train["label"].values
# Scale pixels 0-255 -> 0-1: not "cleaning", just helps the optimizer converge.
X = train.drop(columns="label").values.astype("float32") / 255.0

# --- Authoritative signal: OOF via StratifiedKFold (no groups here) ---
# Same discipline as Trace the Ace: the OOF number is the truth, the LB is noise.
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = cross_val_predict(
    LogisticRegression(max_iter=1000, C=0.1, n_jobs=-1),
    X, y, cv=skf, method="predict", n_jobs=-1,
)
print("v0 OOF accuracy:", round(accuracy_score(y, oof), 4))

# --- Also produce a valid submission on day one: de-risk the plumbing. ---
# A pipeline that can't emit a correct submission format is a bug you want to
# find NOW, not the night before deadline.
clf = LogisticRegression(max_iter=1000, C=0.1, n_jobs=-1).fit(X, y)
test = pd.read_csv("/kaggle/input/competitions/digit-recognizer/test.csv").values.astype("float32") / 255.0
pred = clf.predict(test)


v0 OOF accuracy: 0.9207


In [6]:
pd.DataFrame({"ImageId": np.arange(1, len(pred) + 1), "Label": pred}) \
  .to_csv("submission.csv", index=False)
print("submission.csv written:", len(pred), "rows")

submission.csv written: 28000 rows
